# 03 — Brightness at Constant RMS

Sabit RMS altında yüksek frekans bileşeninin spectral centroid üzerindeki etkisi. İki sinüs bileşeninden oluşan sentetik sinyaller kullanılır.

## Deney ayarları

Alt bileşen 200 Hz, üst bileşen 2000 Hz'dir. `HIGH_RATIOS`, üst bileşenin alt bileşene göre genlik katsayısını belirler. Her sinyalin tüm kayıt RMS'i aynı hedefe ölçeklenir. `TARGET_RMS`, algısal loudness hedefi değildir.

Frame uzunluğu 1200 örnek, yani 50 ms'dir. Her frame iki frekanstan da tam sayıda periyot içerir. `center=False` kenar padding'i eklemez; frame zamanı pencerenin orta noktasıdır.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import importlib.metadata
import json
import platform
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from IPython.display import Audio, display

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SR = 24000
DURATION = 2.0
LOW_HZ = 200.0
HIGH_HZ = 2000.0
HIGH_RATIOS = [0.0, 0.25, 0.5, 1.0, 2.0]
TARGET_RMS = 0.15
FRAME_LENGTH = 1200
HOP_LENGTH = 240
FIGURES = ROOT / "results" / "figures" / "03_brightness_at_constant_rms"
TABLES = ROOT / "results" / "tables" / "03_brightness_at_constant_rms"
EXPERIMENT = ROOT / "experiments" / "03_brightness_at_constant_rms"
AUDIO_OUT = ROOT / "results" / "audio" / "03_brightness_at_constant_rms"
for folder in [FIGURES, TABLES, EXPERIMENT, AUDIO_OUT]:
    folder.mkdir(parents=True, exist_ok=True)


## Sinyal üretimi ve RMS normalizasyonu

$x_a(t)=\sin(2\pi f_{low}t)+a\sin(2\pi f_{high}t)$

$y_a=x_a\cdot\mathrm{RMS}_{target}/\mathrm{RMS}(x_a)$

`a` artırıldıkça üst frekans bileşeninin göreli katkısı artar. Sonraki RMS normalizasyonu toplam sinyal seviyesini eşitler; bileşenler arasındaki genlik oranını değiştirmez. Ölçümler WAV'a yazılmadan önce float64 sinyallerde yapılır.

In [ ]:
assert 0 < LOW_HZ < HIGH_HZ < SR / 2
assert FRAME_LENGTH <= int(SR * DURATION)
t = np.arange(int(SR * DURATION)) / SR
signals = []
rows = []
frame_rms_curves = []
centroid_curves = []
for ratio in HIGH_RATIOS:
    x = np.sin(2*np.pi*LOW_HZ*t) + ratio*np.sin(2*np.pi*HIGH_HZ*t)
    original_rms = np.sqrt(np.mean(x**2))
    y = x * TARGET_RMS / original_rms
    signals.append(y)
    frame_rms = librosa.feature.rms(
        y=y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH,
        center=False, dtype=np.float64)[0]
    D = librosa.stft(y=y, n_fft=FRAME_LENGTH, win_length=FRAME_LENGTH,
                     hop_length=HOP_LENGTH, window="hann", center=False)
    centroid = librosa.feature.spectral_centroid(S=np.abs(D), sr=SR)[0]
    assert len(frame_rms) == len(centroid)
    frame_rms_curves.append(frame_rms)
    centroid_curves.append(centroid)
    rows.append({
        "high_amplitude_ratio": ratio,
        "global_rms": float(np.sqrt(np.mean(y**2))),
        "mean_frame_rms": float(np.mean(frame_rms)),
        "max_frame_rms_target_error": float(np.max(np.abs(frame_rms-TARGET_RMS))),
        "peak": float(np.max(np.abs(y))),
        "mean_centroid_hz": float(np.mean(centroid)),
        "expected_centroid_hz": (LOW_HZ + ratio*HIGH_HZ) / (1 + ratio),
    })
    assert np.max(np.abs(y)) < 1
    sf.write(AUDIO_OUT / f"ratio_{ratio:.2f}.wav", y, SR, subtype="PCM_24")
print("ratio | RMS | centroid (Hz) | expected (Hz)")
for row in rows:
    print(f"{row['high_amplitude_ratio']:4.2f} | {row['global_rms']:.6f} | "
          f"{row['mean_centroid_hz']:.3f} | {row['expected_centroid_hz']:.3f}")


## Centroid karşılaştırması

Bu sinyaller ve analiz ayarları için beklenen centroid:

$C(a)\approx\frac{f_{low}+a f_{high}}{1+a}$

Ağırlıklar genliktir; güç ağırlığı kullanılsaydı `a` yerine `a²` görünürdü. Bu ifade, harmoniklerin FFT kutularıyla hizalandığı mevcut kontrollü örneğe aittir; bütün kayıtlar için genel bir kestirim değildir.

In [ ]:
ratios = np.array([row["high_amplitude_ratio"] for row in rows])
measured = np.array([row["mean_centroid_hz"] for row in rows])
expected = np.array([row["expected_centroid_hz"] for row in rows])
rms_values = np.array([row["global_rms"] for row in rows])
frame_times = (np.arange(len(frame_rms_curves[0])) * HOP_LENGTH + FRAME_LENGTH/2) / SR

assert np.allclose(rms_values, TARGET_RMS, atol=1e-10)
assert all(row["max_frame_rms_target_error"] < 1e-10 for row in rows)
assert np.all(np.diff(measured) > 0)
assert np.allclose(measured, expected, atol=1e-6)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(ratios, measured, "o-", label="Ölçülen centroid")
axes[0].plot(ratios, expected, "x--", label="Beklenen centroid")
axes[0].set(ylabel="Centroid (Hz)", title="Yüksek frekans katkısı ve centroid")
axes[0].legend()
axes[1].plot(ratios, rms_values, "o-", label="Tüm kayıt RMS")
axes[1].axhline(TARGET_RMS, color="gray", linestyle="--", label="Hedef RMS")
axes[1].set(xlabel="Üst bileşenin genlik oranı (a)", ylabel="RMS",
            ylim=(0, 0.3), title="RMS seviyesi")
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURES / "centroid_and_rms.svg", bbox_inches="tight")
plt.show()


## Ses karşılaştırması

Her örnek aynı kayıt RMS'ine sahiptir. `normalize=False` oynatıcının ayrı tepe normalizasyonu yapmasını engeller. Aynı RMS, aynı algısal loudness anlamına gelmez. Tonal bileşenlerin katkısı değiştiğinde tını ve duyulan seviye farklılaşabilir.

In [ ]:
for ratio, y in zip(HIGH_RATIOS, signals):
    print(f"Üst bileşen oranı: {ratio}")
    display(Audio(y, rate=SR, normalize=False))


## Çıktı dosyaları

`comparison.csv` sinyal özetlerini, `frame_features.csv` frame ölçümlerini, `run.json` ayarları ve paket sürümlerini içerir. WAV örnekleri `results/audio/` altında yerel olarak üretilir ve Git'e eklenmez. Aynı çıktı yolları yeniden çalıştırmada güncellenir.

In [ ]:
with (TABLES / "comparison.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
with (TABLES / "frame_features.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["high_amplitude_ratio", "time_s", "rms", "centroid_hz"])
    for ratio, rms_curve, centroid_curve in zip(HIGH_RATIOS, frame_rms_curves, centroid_curves):
        for time, rms, centroid in zip(frame_times, rms_curve, centroid_curve):
            writer.writerow([ratio, float(time), float(rms), float(centroid)])
run = {
    "experiment_id": "03_brightness_at_constant_rms",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "platform": platform.system(),
    "packages": {p: importlib.metadata.version(p) for p in ["numpy", "librosa", "matplotlib", "soundfile"]},
    "settings": {"sr": SR, "duration_s": DURATION, "low_hz": LOW_HZ,
                 "high_hz": HIGH_HZ, "high_amplitude_ratios": HIGH_RATIOS,
                 "target_rms": TARGET_RMS, "frame_length": FRAME_LENGTH,
                 "hop_length": HOP_LENGTH, "center": False,
                 "rms_window": "rectangular", "stft_window": "hann",
                 "centroid_weights": "STFT magnitude", "wav_subtype": "PCM_24",
                 "measurements": "float64 signals before WAV encoding"},
    "signal_float64_sha256": [hashlib.sha256(y.astype("<f8").tobytes()).hexdigest() for y in signals],
    "results": rows,
    "scope": "controlled signal-processing experiment; no perceptual or model evaluation",
}
(EXPERIMENT / "run.json").write_text(
    json.dumps(run, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
print("Kaydedildi:", FIGURES.relative_to(ROOT), TABLES.relative_to(ROOT),
      (EXPERIMENT / "run.json").relative_to(ROOT))
